# 🌸 SaralGati: Fine-Tune Meta-Llama-3.1-8B-Instruct for Cloudflare Workers AI

This notebook trains a custom **LoRA Adapter** on **Meta-Llama-3.1-8B-Instruct** using the **3,000 SaralGati Popular Apps dataset** (WhatsApp, YouTube, Facebook, Phone, Contacts, Photos/Gallery, Messages).

### ⚡ Hardware Requirement:
- Google Colab **Free T4 GPU** (`Runtime` -> `Change runtime type` -> select `T4 GPU`).
- Total training time: **~15 to 18 minutes**.

### 🎯 Output:
1. Creates `saralgati_llama31_8b_lora.zip` (~100MB) and auto-downloads to your computer.
2. 1-Click direct upload to your **Cloudflare Workers AI** account!

## 1. Install Unsloth & Dependencies

In [ ]:
%%capture
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Load Meta-Llama-3.1-8B-Instruct & Configure LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Setup LoRA for Cloudflare compatibility
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("✅ Llama-3.1-8B model & LoRA adapter initialized successfully!")

## 3. Load & Prepare Dataset (3,000 Samples for Popular Apps)

In [ ]:
import json
import os
import urllib.request
from datasets import Dataset

dataset_file = "saralgati_popular_apps_train.jsonl"

# Check if dataset is available locally or download from GitHub
if not os.path.exists(dataset_file):
    try:
        url = "https://raw.githubusercontent.com/ShunyaPulse/SaralGati/main/saralgati_popular_apps_train.jsonl"
        print(f"Downloading dataset from GitHub: {url}...")
        urllib.request.urlretrieve(url, dataset_file)
    except Exception as e:
        print(f"Could not download from GitHub: {e}")
        print("Please upload 'saralgati_popular_apps_train.jsonl' manually using the left sidebar or upload button below.")
        from google.colab import files
        uploaded = files.upload()

data = []
with open(dataset_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

print(f"✅ Successfully loaded {len(data)} high-precision samples from {dataset_file}!")

# Format using official Llama-3.1 chat template
def format_prompts(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompts, batched = True)
print("Sample Formatted Text:\n", dataset[0]["text"][:300], "...")

## 4. Train Model (~15-18 minutes on free T4 GPU)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 15,
        max_steps = 250, # Optimal pass for 3000 high-density samples
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Starting training on GPU...")
trainer_stats = trainer.train()
print("🎉 Training completed successfully!")

## 5. Export Adapter & Patch for Cloudflare Workers AI

In [ ]:
import json
import os
import shutil

output_dir = "saralgati_llama31_8b_lora"
os.makedirs(output_dir, exist_ok = True)

# Save clean LoRA adapter weights (DO NOT merge base model, only ~100MB adapter)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# Patch adapter_config.json with exact base_model_name_or_path for Cloudflare Workers AI
config_path = os.path.join(output_dir, "adapter_config.json")
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

config["base_model_name_or_path"] = "meta-llama/Llama-3.1-8B-Instruct"
config["model_type"] = "llama"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

# Zip for download & upload
zip_filename = "saralgati_llama31_8b_lora"
shutil.make_archive(zip_filename, "zip", output_dir)
print(f"✅ Created archive: {zip_filename}.zip")
print(f"File size: {os.path.getsize(zip_filename + '.zip') / (1024*1024):.2f} MB")

## 6. Download LoRA Zip to Your Computer

In [ ]:
from google.colab import files
files.download("saralgati_llama31_8b_lora.zip")

## 7. 🚀 1-Click Direct Upload to Cloudflare Workers AI

In [ ]:
import requests
import getpass

# Interactive prompt for Cloudflare Credentials
ACCOUNT_ID = input("Enter Cloudflare Account ID (default: 7953d66fe9e6158b01faf0752ae8c841): ").strip() or "7953d66fe9e6158b01faf0752ae8c841"
API_TOKEN = getpass.getpass("Paste Cloudflare API Token: ").strip()
FINE_TUNE_NAME = "saralgati-elder-llama31-8b"
DESCRIPTION = "SaralGati Elder-Friendly Hindi Companion on Llama-3.1-8B-Instruct"

headers = {
    "Authorization": f"Bearer {API_TOKEN}"
}

print(f"Step 1: Checking/Creating fine-tune '{FINE_TUNE_NAME}' on Cloudflare...")
create_url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/ai/finetunes"

# Check if already exists
finetune_id = None
list_res = requests.get(create_url, headers=headers)
if list_res.ok:
    for ft in list_res.json().get("result", []):
        if ft.get("name") == FINE_TUNE_NAME:
            finetune_id = ft.get("id")
            print(f"Found existing fine-tune ID: {finetune_id}")
            break

if not finetune_id:
    create_payload = {
        "name": FINE_TUNE_NAME,
        "description": DESCRIPTION,
        "model": "@cf/meta/llama-guard-3-8b"
    }
    res = requests.post(create_url, headers=headers, json=create_payload)
    if res.ok:
        finetune_id = res.json().get("result", {}).get("id")
        print(f"Created new fine-tune ID: {finetune_id}")

if finetune_id:
    print(f"\nStep 2: Uploading adapter assets to fine-tune '{FINE_TUNE_NAME}' ({finetune_id})...")
    upload_url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/ai/finetunes/{finetune_id}/finetune-assets"

    # Upload adapter_config.json
    with open(f"{output_dir}/adapter_config.json", "rb") as f:
        res1 = requests.post(upload_url, headers=headers, files={"file": ("adapter_config.json", f, "application/json")})
        print("adapter_config.json status:", res1.status_code, res1.text[:100])

    # Upload adapter_model.safetensors
    with open(f"{output_dir}/adapter_model.safetensors", "rb") as f:
        res2 = requests.post(upload_url, headers=headers, files={"file": ("adapter_model.safetensors", f, "application/octet-stream")})
        print("adapter_model.safetensors status:", res2.status_code, res2.text[:100])

    if res1.ok and res2.ok:
        print("\n🎉🎉 SUCCESS! LoRA adapter is fully uploaded and live on Cloudflare Workers AI!")
    else:
        print("\n⚠️ Upload finished with warnings. Check responses above.")
else:
    print("❌ Could not obtain fine-tune ID. Check API token permissions.")